# Jets: log p, full vs noscalar (5 seeds each)

Data from `python jets/logp_jets.py` (Jean Zay) -> `figures/logp_jets/logp_jets.npz`; this notebook only reads it
(seconds; runs locally too after pulling that one file). Run from `jets/`.

- Theta = Theta_reg(t = 1) at the cross-seed lambda (1e-2 for both arms, window t >= 0.99), unconstrained.
- log p(x) = sum over families of C_F(x) = Theta_F . phi_F(x), **up to the unknown constant -log Z** of each seed:
  only differences under the same Theta are exact (rare - typical, held-out - training, spreads). Mean C_F of one
  family alone also depends on where phi_F is centred; compare D_F and the spreads, not the raw levels.
- rare = max |lag-1 increment| above the 95th percentile of the training data.
- **noscalar caveat:** L_6 filter 1 > 0 in 5/5 seeds at lambda 1e-2 (density not normalizable along that direction).

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
from IPython.display import display
import logp_jets as lj
pd.set_option('display.float_format', '{:,.3g}'.format, 'display.width', 220)

runs, inc = lj.load()
rare, thr = lj.rare_masks(inc, P=95)
for a, r in runs.items():
    print(f"{a}: seeds {r['seeds']}, lam {r['lam']:g}, families {r['families'][:-1]}")
print(f"rare threshold {thr:.3g}: train {int(rare['train'].sum())} / {rare['train'].size}, "
      f"held-out {int(rare['heldout'].sum())} / {rare['heldout'].size}")

## 1. Distribution of log p (training data)
Each seed centred on its own median (removes its -log Z). Left panels: the 5 seeds of an arm (rare signals dashed);
right: both arms pooled over seeds. Look at the shape: width, left tail (the rare, unlikely signals).

In [ ]:
fig = lj.plot_distribution(runs, rare)

In [ ]:
# shape numbers per arm, mean (sd) over seeds: spread, skewness, left tail
from scipy import stats
rows = []
for a, r in runs.items():
    v = [r['C']['train'][i, -1] for i in range(len(r['seeds']))]
    f = lambda fn: (np.mean([fn(x) for x in v]), np.std([fn(x) for x in v], ddof=1))
    for name, fn in (('std', np.std), ('skew', stats.skew), ('excess kurtosis', stats.kurtosis),
                     ('median - q0.001', lambda x: np.median(x) - np.quantile(x, 1e-3))):
        m, s = f(fn)
        rows.append(dict(arm=a, stat=name, mean=m, sd_over_seeds=s))
display(pd.DataFrame(rows).pivot(index='stat', columns='arm'))

## 2. log p per potential family, full vs noscalar
Panels: mean C_F over typical / rare signals; D_F = rare - typical (how much each family makes rare signals less
likely); share D_F / D_TOTAL; spread of C_F across signals. Error bars = sd over the 5 seeds.

In [ ]:
fig, T = lj.plot_family_bars(runs, rare)

In [ ]:
# seed robustness (point a): D_F and share as mean +- sd over seeds; rel_sd = sd / |mean| of D_F
T['rel_sd_D'] = T['D_sd'] / T['D'].abs()
display(T.pivot(index='family', columns='arm', values=['D', 'D_sd', 'rel_sd_D', 'share', 'share_sd'])
         .reindex(list(dict.fromkeys(T.family))))

## 3. Held-out data (point c)
Windows of the record never used in training and more than `gap` windows away from every training window, standardized
with the training mean / std. Same Theta on both sets, so the constant cancels: a held-out distribution shifted to the
left (negative TOTAL shift) = held-out signals less likely than training ones, i.e. overfitting (or held-out windows
genuinely different). The scale for "large": the std of log p across signals (section 1, ~ `std` row).

In [ ]:
fig, H = lj.plot_heldout(runs, rare)
display(H.pivot(index='family', columns='arm', values=['shift', 'shift_sd', 'shift_over_std'])
         .reindex(list(dict.fromkeys(H.family))))

In [ ]:
# the bar chart of section 2 on the held-out data: does the family decomposition hold on unseen signals?
fig, Th = lj.plot_family_bars(runs, rare, data_set='heldout')